# Scenario Planner — Interactive Analytics Demo

**Macro-Driven Equity Risk Platform**

This notebook demonstrates the full analytical output of the Medallion Pipeline using synthetic data that mirrors the real pipeline's Master Table format. No API keys required — everything runs offline.

---

**Contents**
1. Project Architecture Visualization
2. Monte Carlo GBM Simulation
3. Sensitivity Regression (OLS + Ridge)
4. Correlation Matrix (Risk Diversification)
5. Stress Test — Black Swan Scenarios
6. ARIMA Forecasting
7. Lag Analysis — Delayed Macro Impact
8. Code Quality Dashboard (Ruff / MyPy / Pytest)

In [ ]:
import importlib

REQUIRED = ["numpy", "pandas", "matplotlib", "seaborn", "scipy", "statsmodels", "sklearn"]
missing = [pkg for pkg in REQUIRED if importlib.util.find_spec(pkg) is None]
if missing:
    print(f"⚠️  Missing packages: {missing}\nRun: poetry install")
else:
    print("✅  All dependencies available — ready to run.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# ── Plot style ────────────────────────────────────────────────────────────────
sns.set_theme(style="darkgrid", palette="muted", font_scale=1.1)
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "figure.facecolor": "#0f0f0f",
    "axes.facecolor": "#1a1a2e",
    "text.color": "white",
    "axes.labelcolor": "white",
    "xtick.color": "white",
    "ytick.color": "white",
    "axes.edgecolor": "#444",
    "grid.color": "#333",
})
TICKERS = ["AAPL", "TSLA", "MSFT", "WMT", "XOM"]
print("📦 Imports OK | Tickers:", TICKERS)

---
## 1. Project Architecture — Medallion Pipeline

```
RAW SOURCES          BRONZE                SILVER               GOLD
─────────────        ─────────────         ─────────────        ───────────────────
Yahoo Finance   ──►  Raw Parquet     ──►   Schema Valid   ──►   Master Feature Table
FRED API        ──►  + Metadata            Winsorize            log_return engineered
World Bank      ──►  + Catalog             Standardize          ──► AnalysisSuite ──►
                     Retry Logic           ZSTD Parquet              Monte Carlo
                     Parallel I/O                                     OLS / Ridge
                                                                      ARIMA
                                                                      Stress Test
```

**Why Log Returns?** Raw price differences are scale-dependent and non-stationary.
$r_t = \ln(P_t / P_{t-1})$ is additive over time, approximately normal, and comparable
across tickers regardless of absolute price level.

**Why Monte Carlo GBM?** A single ARIMA forecast gives one path. GBM with $N=10{,}000$
paths gives a *full probability distribution* — we can quote VaR at any confidence level.

In [ ]:
"""
Synthetic Master Table — mirrors the real GoldLayer output so this
notebook runs without API keys.  All distributions are calibrated
against realistic equity + macro ranges (2016-2026 AAPL/TSLA data).
"""
np.random.seed(42)
N = 504  # ~2 years of trading days

dates = pd.date_range("2024-01-01", periods=N, freq="B")

rows = []
params = {
    "AAPL":  dict(mu=0.00045, sigma=0.0155, start=185.0),
    "TSLA":  dict(mu=0.00030, sigma=0.0310, start=240.0),
    "MSFT":  dict(mu=0.00040, sigma=0.0140, start=370.0),
    "WMT":   dict(mu=0.00025, sigma=0.0095, start=165.0),
    "XOM":   dict(mu=0.00020, sigma=0.0130, start=110.0),
}

for ticker, p in params.items():
    log_ret = np.random.normal(p["mu"], p["sigma"], N)
    prices = p["start"] * np.exp(np.cumsum(log_ret))
    rows.append(pd.DataFrame({
        "date": dates,
        "ticker": ticker,
        "close": prices,
        "log_return": log_ret,
    }))

master_df = pd.concat(rows, ignore_index=True)

# Macro factors (FRED-style: monthly, forward-filled)
master_df["inflation"]     = np.interp(
    np.arange(N), np.linspace(0, N-1, 60),
    0.03 + 0.04 * np.random.randn(60).cumsum() / 20
)
master_df["energy_index"]  = np.interp(
    np.arange(N), np.linspace(0, N-1, 60),
    100 + 20 * np.random.randn(60).cumsum() / 10
)

print(f"Master Table: {master_df.shape}  |  Tickers: {master_df.ticker.unique().tolist()}")
master_df.head(3)

---
## 2. Monte Carlo GBM — 10 000-Path Price Simulation

$dS_t = \mu S_t \, dt + \sigma S_t \, dW_t$

We simulate **10,000 independent paths** for AAPL over 252 trading days.
The fan chart shows the median + confidence bands (5th, 25th, 75th, 95th percentile).

In [ ]:
TICKER = "AAPL"
DAYS, ITERATIONS = 252, 10_000

aapl_df = master_df[master_df["ticker"] == TICKER].copy()
returns = aapl_df["log_return"].dropna().values
mu, sigma = returns.mean(), returns.std()
last_price = aapl_df["close"].iloc[-1]

# Vectorised GBM
shocks = np.exp(
    (mu - 0.5 * sigma**2) + sigma * np.random.normal(0, 1, (DAYS, ITERATIONS))
)
paths = last_price * shocks.cumprod(axis=0)

# Percentile bands
p5, p25, p50, p75, p95 = [np.percentile(paths, q, axis=1) for q in (5, 25, 50, 75, 95)]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle(f"Monte Carlo GBM — {TICKER}  |  {ITERATIONS:,} paths × {DAYS} days",
             color="white", fontsize=14, fontweight="bold")

# Fan chart
ax = axes[0]
ax.fill_between(range(DAYS), p5,  p95, alpha=0.15, color="#60a0ff", label="5th–95th %ile")
ax.fill_between(range(DAYS), p25, p75, alpha=0.30, color="#60a0ff", label="25th–75th %ile")
ax.plot(p50, color="#60a0ff", linewidth=2, label="Median")
ax.axhline(last_price, color="#ff6060", linestyle="--", linewidth=1, label=f"Last Price ${last_price:.0f}")
ax.set_xlabel("Trading Days")
ax.set_ylabel("Simulated Price ($)")
ax.set_title("Price Path Fan Chart")
ax.legend(fontsize=9)

# Terminal price distribution
ax2 = axes[1]
terminal = paths[-1, :]
ax2.hist(terminal, bins=80, color="#60a0ff", alpha=0.75, density=True, label="Terminal Distribution")
ax2.axvline(np.percentile(terminal, 5),  color="#ff4444", linestyle="--", linewidth=1.5, label="VaR 95%")
ax2.axvline(np.percentile(terminal, 50), color="#44ff88", linestyle="-",  linewidth=2,   label="Median")
ax2.set_xlabel("Terminal Price ($)")
ax2.set_title("Terminal Price Distribution (Day 252)")
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

var_95 = last_price - np.percentile(terminal, 5)
print(f"📊  Drift μ = {mu:.5f}  |  Volatility σ = {sigma:.5f}")
print(f"📉  VaR (95%, 1-year) = ${var_95:.2f}  ({var_95/last_price*100:.1f}% of current price)")

---
## 3. Sensitivity Regression — OLS & Ridge

How sensitive are AAPL's log-returns to inflation and energy prices?

| Model | Use case |
|-------|----------|
| **OLS** | Full statistical report: t-stats, p-values, R², confidence intervals |
| **Ridge** | Collinear factors (L2 regularisation prevents coefficient blow-up) |

In [ ]:
import statsmodels.api as sm
from sklearn.linear_model import Ridge

aapl = master_df[master_df["ticker"] == "AAPL"].dropna(subset=["log_return", "inflation", "energy_index"])
FACTORS = ["inflation", "energy_index"]
Y = aapl["log_return"]
X_ols = sm.add_constant(aapl[FACTORS])

# ── OLS ───────────────────────────────────────────────────────────────────────
ols_model = sm.OLS(Y, X_ols).fit()
print("═" * 60)
print("OLS SUMMARY")
print("═" * 60)
print(ols_model.summary().tables[1])

# ── Ridge ─────────────────────────────────────────────────────────────────────
ridge = Ridge(alpha=0.1)
ridge.fit(aapl[FACTORS], Y)
ridge_coefs = dict(zip(FACTORS, ridge.coef_))
print(f"\nRidge Coefficients: {ridge_coefs}")
print(f"Ridge Intercept:    {ridge.intercept_:.6f}")

# ── Visual comparison ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle("Sensitivity Regression — Macro Factor Coefficients", color="white", fontsize=13, fontweight="bold")

colors = ["#60a0ff", "#ff9944"]
for ax, factor, color in zip(axes, FACTORS, colors):
    x = aapl[factor]
    y = Y
    ax.scatter(x, y, alpha=0.25, s=10, color=color)
    m, b = np.polyfit(x, y, 1)
    x_line = np.linspace(x.min(), x.max(), 100)
    ax.plot(x_line, m * x_line + b, color="white", linewidth=2)
    ax.set_xlabel(factor.replace("_", " ").title())
    ax.set_ylabel("AAPL log_return")
    ax.set_title(f"β({factor}) = {ols_model.params[factor]:.4f}  (p={ols_model.pvalues[factor]:.3f})")

plt.tight_layout()
plt.show()

---
## 4. Correlation Matrix — Risk Diversification Heatmap

A value near **+1** means assets move together (concentrated risk).  
Near **0** = diversification benefit.  
Near **−1** = natural hedge.

In [ ]:
# Pivot: one log_return column per ticker + macro factors
pivot = master_df.pivot_table(index="date", columns="ticker", values="log_return")
pivot["inflation"]    = master_df.drop_duplicates("date").set_index("date")["inflation"]
pivot["energy_index"] = master_df.drop_duplicates("date").set_index("date")["energy_index"]

corr = pivot.corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
cmap = sns.diverging_palette(220, 10, as_cmap=True)
sns.heatmap(
    corr, mask=mask, cmap=cmap, vmin=-1, vmax=1, center=0,
    annot=True, fmt=".2f", annot_kws={"size": 10},
    square=True, linewidths=0.5, cbar_kws={"shrink": 0.8},
    ax=ax
)
ax.set_title("Cross-Asset & Macro Correlation Matrix", color="white", fontsize=14, fontweight="bold", pad=15)
plt.tight_layout()
plt.show()

# Highlight strongest macro-equity links
print("\n🔍  Strongest macro–equity correlations:")
macro_corr = corr.loc[TICKERS, ["inflation", "energy_index"]].abs()
print(macro_corr.sort_values("inflation", ascending=False).to_string())

---
## 5. Stress Test — Black Swan Scenarios

Each bar shows the **linear regression–predicted impact** on log-returns if
a macro factor receives an instantaneous shock (e.g., inflation +10%).

> Interpretation: $\text{impact} = \hat{\beta}_\text{factor} \times \Delta_\text{shock}$

In [ ]:
SHOCK_MAP = {"inflation": 0.10, "energy_index": 0.20}  # +10% inflation, +20% energy

results = {}
for ticker in TICKERS:
    df_t = master_df[master_df["ticker"] == ticker].dropna(subset=["log_return"] + list(SHOCK_MAP.keys()))
    ticker_impacts = {}
    for factor, shock in SHOCK_MAP.items():
        model = sm.OLS(df_t["log_return"], sm.add_constant(df_t[factor])).fit()
        ticker_impacts[factor] = model.params[factor] * shock
    results[ticker] = ticker_impacts

results_df = pd.DataFrame(results).T  # tickers × factors

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
fig.suptitle("Stress Test — Predicted Return Impact by Ticker", color="white", fontsize=13, fontweight="bold")

for ax, (factor, shock) in zip(axes, SHOCK_MAP.items()):
    vals = results_df[factor]
    colors = ["#ff4444" if v < 0 else "#44ff88" for v in vals]
    bars = ax.barh(vals.index, vals.values * 100, color=colors, alpha=0.85, height=0.5)
    ax.axvline(0, color="white", linewidth=0.8, linestyle="--")
    ax.set_xlabel("Predicted Return Impact (%)")
    ax.set_title(f"{factor.replace('_',' ').title()}  Δ={shock*100:.0f}%")
    for bar, val in zip(bars, vals.values):
        ax.text(val * 100 + (0.002 if val >= 0 else -0.002), bar.get_y() + bar.get_height()/2,
                f"{val*100:+.3f}%", va="center", ha="left" if val >= 0 else "right",
                fontsize=9, color="white")

plt.tight_layout()
plt.show()

print("\n📋  Stress Test Summary:")
print((results_df * 100).applymap(lambda x: f"{x:+.3f}%").to_string())

---
## 6. ARIMA Forecasting

ARIMA(5,1,0) projects the next 30 trading days of AAPL log-returns.
The 95% confidence interval widens as uncertainty compounds over time.

In [ ]:
import warnings

from statsmodels.tsa.arima.model import ARIMA

STEPS = 30
aapl_ts = (
    master_df[master_df["ticker"] == "AAPL"]
    .set_index("date")["log_return"]
    .dropna()
)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    arima_model = ARIMA(aapl_ts, order=(5, 1, 0)).fit()
    forecast_result = arima_model.get_forecast(steps=STEPS)

fc_mean = forecast_result.predicted_mean
fc_ci   = forecast_result.conf_int(alpha=0.05)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("ARIMA(5,1,0) Forecast — AAPL Log-Returns", color="white", fontsize=13, fontweight="bold")

# Recent history + forecast
ax = axes[0]
history = aapl_ts.iloc[-60:]
ax.plot(history.index, history.values, color="#60a0ff", linewidth=1.5, label="Historical")
ax.plot(fc_mean.index, fc_mean.values, color="#ffdd44", linewidth=2, label="Forecast")
ax.fill_between(fc_mean.index, fc_ci.iloc[:, 0], fc_ci.iloc[:, 1],
                alpha=0.25, color="#ffdd44", label="95% CI")
ax.axvline(aapl_ts.index[-1], color="white", linestyle="--", linewidth=0.8)
ax.set_xlabel("Date")
ax.set_ylabel("Log Return")
ax.set_title("30-Day Forward Projection")
ax.legend(fontsize=9)
ax.tick_params(axis="x", rotation=30)

# Residual diagnostics
ax2 = axes[1]
residuals = arima_model.resid.iloc[-120:]
ax2.plot(residuals.index, residuals.values, color="#ff9944", linewidth=0.8, alpha=0.7)
ax2.axhline(0, color="white", linestyle="--", linewidth=0.8)
ax2.set_xlabel("Date")
ax2.set_ylabel("Residual")
ax2.set_title("Model Residuals (last 120 days)")
ax2.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

print(f"📈  AIC: {arima_model.aic:.2f}  |  BIC: {arima_model.bic:.2f}")
print(f"📊  30-day forecast μ: {fc_mean.mean():.5f}  |  σ: {fc_mean.std():.5f}")

---
## 7. Lag Analysis — Delayed Macro Impact

$\rho(k) = \text{Corr}(X_t,\; X_{t-k})$

High autocorrelation at lag *k* means today's value is predictable from *k* periods ago.
This is used to select the AR order `p` in ARIMA.

In [ ]:
LAGS = 10
macro_series = master_df.drop_duplicates("date").set_index("date")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle("Lag Autocorrelation — Macro Factors", color="white", fontsize=13, fontweight="bold")

for ax, col, color in zip(axes, ["inflation", "energy_index"], ["#60a0ff", "#ff9944"]):
    series = macro_series[col].dropna()
    lag_corrs = {f"lag_{i}": series.corr(series.shift(i)) for i in range(1, LAGS + 1)}

    ax.bar(list(lag_corrs.keys()), list(lag_corrs.values()), color=color, alpha=0.8)
    ax.axhline(0, color="white", linewidth=0.8)
    ax.axhline( 1.96 / np.sqrt(len(series)), color="white", linestyle="--", linewidth=0.8, label="95% significance")
    ax.axhline(-1.96 / np.sqrt(len(series)), color="white", linestyle="--", linewidth=0.8)
    ax.set_title(col.replace("_", " ").title())
    ax.set_xlabel("Lag")
    ax.set_ylabel("Autocorrelation ρ")
    ax.tick_params(axis="x", rotation=45)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print("\n🔍  Inflation lag correlations:")
for k, v in lag_corrs.items():
    print(f"  {k}: {v:.4f}")

---
## 8. Code Quality Dashboard

Runs `ruff`, `mypy`, and `pytest` via subprocess and renders a pass/fail table.

In [ ]:
import subprocess
from pathlib import Path

PROJECT_ROOT = Path(__file__).parents[1] if "__file__" in dir() else Path.cwd().parent

CHECKS = [
    ("Ruff  (lint)",  ["poetry", "run", "ruff",  "check", "."]),
    ("Ruff  (format check)", ["poetry", "run", "ruff",  "format", "--check", "."]),
    ("MyPy  (types)", ["poetry", "run", "mypy",  "src/"]),
    ("Pytest (tests)", ["poetry", "run", "pytest", "--tb=line", "-q"]),
]

print(f"{'Check':<25} {'Status':<10} {'Details'}")
print("─" * 70)

all_passed = True
for name, cmd in CHECKS:
    try:
        result = subprocess.run(
            cmd, cwd=str(PROJECT_ROOT),
            capture_output=True, text=True, timeout=120
        )
        passed = result.returncode == 0
        status = "✅ PASS" if passed else "❌ FAIL"
        # Extract last non-empty line as summary
        output = (result.stdout + result.stderr).strip().splitlines()
        summary = next((l for l in reversed(output) if l.strip()), "")[:60]
        print(f"{name:<25} {status:<10} {summary}")
        if not passed:
            all_passed = False
    except FileNotFoundError:
        print(f"{name:<25} ⚠️  SKIP   poetry not found — run outside venv")
    except subprocess.TimeoutExpired:
        print(f"{name:<25} ⏱️  TIMEOUT")

print("─" * 70)
print("🏆  ALL CHECKS PASSED — Ready for git push!" if all_passed else "⚠️  Fix failing checks before pushing.")